In [1]:
!pip install face_recognition
!pip install opencv-python
!pip install pandas
!pip install --upgrade dlib face_recognition

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 9.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for face-recognition-models: filename=face_recognition_models-0.3.0-py2.py3-none-any.whl size=100566166 sha256=4ce6be1d5dc7ec8cfc8ff6fc81c67b6e611ff713b996ea50435f60f45c8beffb
  Stored in directory: /root/.cache/pip/wheels/8f/47/c8/f44c5aebb7507f7c8a2c0bd23151d732d0f0bd6884ad4ac635
Successfully built face-recognition-models
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 54.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for dlib: filename=dlib-20.0.1-cp312-cp312-linux_x86_64.whl size=4283637 sha256=494c0d8393dc3f04e302a3dd4db7453e200424a1b6e61a0a13d64c5e18952ee6
  Stored in directory: /root/.cache/pip/wheels/0d/a3/e4/89c1a393cbb86433554f89465e1f2b4c647bca9f256b454b62
Successfully built dlib
  Attempting uninstall: dlib
 

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import face_recognition
import pickle
import cv2

def train_and_save_model(student_images_folder, model_file='trained_model.pickle'):
    student_encodings = []
    student_names = []

    for filename in os.listdir(student_images_folder):
        if filename.endswith(('.jpg', '.png')):
            img_path = os.path.join(student_images_folder, filename)
            image = face_recognition.load_image_file(img_path)

            image = cv2.resize(image, (0, 0), fx=0.5, fy=0.5)

            encoding = face_recognition.face_encodings(image)[0]
            student_encodings.append(encoding)
            student_names.append(os.path.splitext(filename)[0])

    with open(model_file, 'wb') as f:
        pickle.dump((student_encodings, student_names), f)

    print(f'Model trained and saved to {model_file}')

student_images_folder = '/content/drive/MyDrive/Smart Attandance System/Images'
train_and_save_model(student_images_folder)


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (108000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Model trained and saved to trained_model.pickle


In [4]:
import cv2
import face_recognition
import pandas as pd
import pickle
from datetime import datetime

def load_trained_model(model_file='trained_model.pickle'):
    with open(model_file, 'rb') as f:
        student_encodings, student_names = pickle.load(f)
    return student_encodings, student_names

def mark_attendance(student_names, recognized_names):
    now = datetime.now()
    date = now.strftime('%d-%m-%Y')
    time = now.strftime('%H:%M')

    attendance = pd.DataFrame(student_names, columns=['Student Name'])
    attendance['Date'] = date
    attendance['Time'] = time
    attendance['Status'] = ['Present' if name in recognized_names else 'Absent' for name in student_names]

    return attendance

def recognize_faces_in_group_photo(group_photo_path, student_encodings, student_names):
    group_photo = face_recognition.load_image_file(group_photo_path)

    small_frame = cv2.resize(group_photo, (0, 0), fx=0.5, fy=0.5)

    face_locations = face_recognition.face_locations(small_frame)

    face_locations = [(top*2, right*2, bottom*2, left*2) for top, right, bottom, left in face_locations]

    face_encodings = face_recognition.face_encodings(group_photo, face_locations)

    recognized_names = []

    for encoding in face_encodings:
        matches = face_recognition.compare_faces(student_encodings, encoding)
        face_distances = face_recognition.face_distance(student_encodings, encoding)
        best_match_index = face_distances.argmin()

        if matches[best_match_index]:
            recognized_names.append(student_names[best_match_index])

    return recognized_names

def main(group_photo_path, model_file='trained_model.pickle'):
    student_encodings, student_names = load_trained_model(model_file)

    recognized_names = recognize_faces_in_group_photo(group_photo_path, student_encodings, student_names)

    attendance = mark_attendance(student_names, recognized_names)

    print(attendance)
    attendance.to_csv('attendance.csv', index=False)
    print("Attendance saved to 'attendance.csv'.")

group_photo_path = '/content/drive/MyDrive/Smart Attandance System/IMG20241023152326.jpg'
main(group_photo_path)


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (108000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


        Student Name        Date   Time   Status
0       Shivam_Kumar  19-04-2026  18:47  Present
1           Hans_Raj  19-04-2026  18:47  Present
2  Amritangshu_Singh  19-04-2026  18:47  Present
3         Khyati_Raj  19-04-2026  18:47  Present
4        Nawneet_Raj  19-04-2026  18:47  Present
Attendance saved to 'attendance.csv'.
